# DIMAGGI Tool Guard — Runtime Policy Firewall for Gemma 4 Agents

> **DIMAGGI Tool Guard · Clinical AI Governance**

**Competition:** [Kaggle × Google DeepMind Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon) — Safety & Trust track.

**Live Demo:** [https://dimaggi.ai/tool-guard/](https://dimaggi.ai/tool-guard/)  ·  **GitHub:** [dimaggi-ai/health-tool-guard](https://github.com/dimaggi-ai/health-tool-guard)  ·  **License:** [AGPL-3.0](https://github.com/dimaggi-ai/health-tool-guard/blob/main/LICENSE)

This notebook is a **self-contained, reproducible** walkthrough of the Tool Guard policy engine using Gemma 4 E4B. Three scenarios run end-to-end with real model inference (when the Kaggle Gemma 4 dataset is attached) or scripted decisions (when offline), producing a tamper-evident SHA-256 audit chain.

---

## What

**Tool Guard** is a runtime policy firewall that sits between an AI agent and its tool calls. Before any tool executes — prescribe_medication, send_report, fetch_patient_record — Tool Guard intercepts the request and runs three actions on every call:

- **⚖ DECIDE** — evaluate against a configurable policy set (deterministic in <15 ms; Gemma 4 hybrid in 700–900 ms).
- **⛓ AUDIT** — append a SHA-256 + HMAC-signed record to a tamper-evident chain.
- **🙋 ESCALATE** — route to a human reviewer when Gemma 4 cannot be trusted (unsupported language, low confidence, ambiguous safety call).

## Why

AI agents in healthcare, finance, and government operate across linguistic and cultural boundaries where a single bad tool call can cause patient harm, violate GDPR/HIPAA, or breach cross-border data rules. Deterministic rules alone cannot catch nuanced clinical contraindications (e.g., NSAID cross-reactivity with aspirin allergy). Gemma 4's reasoning bridges that gap — and Tool Guard makes Gemma's decisions auditable, escalatable, and reversible.

## How

```
Agent Tool Call
      │
      ▼
┌─────────────────────────────────────────────────┐
│              TOOL GUARD ENGINE                  │
│  ┌─────────────┐    ┌──────────────────────┐   │
│  │Deterministic│    │  Gemma 4 Hybrid      │   │
│  │Rules (<15ms)│───▶│  Reasoning (~800ms)  │   │
│  │PII / Lang   │    │  Clinical / Complex  │   │
│  └─────────────┘    └──────────────────────┘   │
│              │                                  │
│              ▼                                  │
│  ┌─────────────────────────────────────────┐   │
│  │  DECIDE: ALLOW · REDACT · DENY · ESC.   │   │
│  └─────────────────────────────────────────┘   │
│              │                                  │
│              ▼                                  │
│  ┌─────────────────────────────────────────┐   │
│  │  SHA-256 Hash-Chained · HMAC-Signed     │   │
│  │  Independently verifiable audit record  │   │
│  └─────────────────────────────────────────┘   │
└─────────────────────────────────────────────────┘
```

**Three scenarios** demonstrate the full capability:
1. **NSAID/Arabic Contraindication** — Gemma 4 hybrid reasoning catches COX-1 cross-reactivity → DENY
2. **Swahili PII Redact** — Deterministic field-scrub in <12 ms → REDACT_AND_CONTINUE
3. **Luganda Language Mismatch** — Deterministic enforce → DENY; shadow mode → ALLOW + near-miss banner

The submission video (linked from the README) is a complementary view focused on **Luganda DENY + Acholi ESCALATE** — chosen for multimodal impact, not to replace the breadth shown below.

---

> **Built with Gemma 4** under the [Gemma Terms of Use](https://ai.google.dev/gemma/terms). FDA drug-label grounding via [openFDA](https://open.fda.gov/) (public domain).

In [ ]:
!pip install -q transformers accelerate bitsandbytes langdetect kagglehub
print("✓ All dependencies ready")

In [ ]:
import os, sys

# Load Gemma 4 E4B — Kaggle hosted model
MODEL_PATH = None

# Option 1: Kaggle dataset input (attach dataset in notebook settings)
kaggle_paths = [
    "/kaggle/input/gemma-4-e4b-it",
    "/kaggle/input/gemma4/transformers/gemma-4-e4b-it/1",
]
for p in kaggle_paths:
    if os.path.exists(p):
        MODEL_PATH = p
        print(f"Found model at: {MODEL_PATH}")
        break

# Option 2: kagglehub download
if MODEL_PATH is None:
    try:
        import kagglehub
        MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e4b-it")
        print(f"Downloaded model to: {MODEL_PATH}")
    except Exception as e:
        print(f"kagglehub download failed: {e}")

if MODEL_PATH is None:
    raise RuntimeError("Could not find Gemma 4 model. Attach the gemma-4-e4b-it dataset as input.")

print(f"\nUsing model from: {MODEL_PATH}")


Loading Gemma 4 E4B-IT model...


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  TOOL GUARD ENGINE
# ═══════════════════════════════════════════════════════════════════════════════

CLINICAL_GUIDELINES = """FDA Ibuprofen OTC label: may cause severe allergic reaction especially in aspirin-allergic patients. FDA Naproxen SPL Section 4: contraindicated in patients with history of asthma, urticaria, or allergic reactions after aspirin or NSAIDs. MECHANISM: COX-1 inhibition — all NSAIDs cross-react with aspirin allergy. Safe alternative: paracetamol (acetaminophen) does NOT inhibit COX-1 and does NOT cross-react. AERD: 7% of asthmatics, up to 30-40% with nasal polyposis [PMC3005316, ACC 2024]."""


def _gemma4_call(prompt: str, max_new_tokens: int = 600, temperature: float = 0.1) -> str:
    """Call Gemma 4 with the instruction-tuned chat template."""
    full_prompt = (
        f"<start_of_turn>user\n{prompt}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    inputs = tokenizer(full_prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def _extract_json(text: str) -> dict:
    """Extract first JSON object from a string that may contain prose."""
    # Try to find ```json ... ``` block
    m = re.search(r"```json\s*([\s\S]+?)```", text)
    if m:
        try:
            return json.loads(m.group(1).strip())
        except json.JSONDecodeError:
            pass
    # Try to find bare { ... } block
    m = re.search(r"(\{[\s\S]+\})", text)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    return {"decision": "DENY", "policy_triggered": "parse-error",
            "explanation": text[:300], "reasoning": "", "why_hybrid": ""}


# ── PII field detector ────────────────────────────────────────────────────────
PII_FIELDS = {
    "patient_name", "name", "full_name", "dob", "date_of_birth",
    "id_number", "national_id", "passport", "ssn", "phone",
    "email", "address", "mrn", "medical_record_number",
}


def _redact_params(params: dict) -> Tuple[dict, List[str]]:
    """Redact PII fields. Returns (redacted_params, list_of_redacted_keys)."""
    redacted = {}
    fields_hit = []
    for k, v in params.items():
        if k.lower() in PII_FIELDS:
            redacted[k] = "[REDACTED]"
            fields_hit.append(k)
        else:
            redacted[k] = v
    return redacted, fields_hit


# ── Language detection helpers ────────────────────────────────────────────────
LANG_NAME = {
    "en": "English", "ar": "Arabic", "sw": "Swahili",
    "lg": "Luganda", "fr": "French", "ha": "Hausa",
    "yo": "Yoruba", "am": "Amharic",
}


def _lang_name(code: str) -> str:
    return LANG_NAME.get(code, code.upper())


# ════════════════════════════════════════════════════════════════════════════════
#  AuditChain — tamper-evident SHA-256 hash chain
# ════════════════════════════════════════════════════════════════════════════════

class AuditChain:
    """Immutable, tamper-evident audit log using SHA-256 hash chaining."""

    def __init__(self, secret: bytes = HMAC_SECRET):
        self.secret = secret
        self.records: List[dict] = []
        self._prev_hash = "0" * 64  # genesis block

    def add_record(
        self,
        tool: str,
        decision: str,
        eval_path: str,
        latency_ms: float,
        policy: str,
        explanation: str,
        modality: str = "text",
        extra: Optional[dict] = None,
    ) -> dict:
        record_id = str(uuid.uuid4())
        nonce = secrets.token_hex(16)
        ts = datetime.datetime.utcnow().isoformat() + "Z"

        base = {
            "id": record_id,
            "timestamp": ts,
            "tool": tool,
            "decision": decision,
            "eval_path": eval_path,
            "latency_ms": round(latency_ms, 2),
            "policy": policy,
            "explanation": explanation,
            "nonce": nonce,
            "modality": modality,
            "prev_hash": self._prev_hash,
        }
        if extra:
            base.update(extra)

        # HMAC-SHA256 over canonical JSON (sorted keys)
        canonical = json.dumps(base, sort_keys=True, ensure_ascii=False)
        mac = hmac.new(self.secret, canonical.encode(), hashlib.sha256).hexdigest()
        base["hmac"] = mac

        # SHA-256 of (prev_hash + canonical + hmac)
        chain_input = (self._prev_hash + canonical + mac).encode()
        block_hash = hashlib.sha256(chain_input).hexdigest()
        base["hash"] = block_hash

        self._prev_hash = block_hash
        self.records.append(base)
        return base

    def verify(self) -> Tuple[bool, int]:
        """Re-compute every hash and HMAC. Returns (ok, count_verified)."""
        prev = "0" * 64
        for i, rec in enumerate(self.records):
            r = {k: v for k, v in rec.items() if k not in ("hmac", "hash")}
            canonical = json.dumps(r, sort_keys=True, ensure_ascii=False)
            expected_mac = hmac.new(self.secret, canonical.encode(), hashlib.sha256).hexdigest()
            if not hmac.compare_digest(expected_mac, rec["hmac"]):
                return False, i
            chain_input = (prev + canonical + rec["hmac"]).encode()
            expected_hash = hashlib.sha256(chain_input).hexdigest()
            if expected_hash != rec["hash"]:
                return False, i
            prev = rec["hash"]
        return True, len(self.records)

    def as_jsonl(self) -> str:
        return "\n".join(json.dumps(r, ensure_ascii=False) for r in self.records)


# ════════════════════════════════════════════════════════════════════════════════
#  ToolGuard — policy evaluation engine
# ════════════════════════════════════════════════════════════════════════════════

class ToolGuard:
    """Runtime policy firewall for AI agent tool calls."""

    def __init__(self, policies: List[dict], audit: Optional[AuditChain] = None, mode: str = "enforce"):
        """
        mode: 'enforce' → actually deny/redact
              'shadow'  → allow everything but record near-misses
        """
        self.policies = policies
        self.audit = audit or AuditChain()
        self.mode = mode  # 'enforce' | 'shadow'

    def evaluate(
        self,
        tool: str,
        params: dict,
        context: Optional[dict] = None,
    ) -> dict:
        """Evaluate a tool call against all policies. Returns a decision record."""
        context = context or {}
        t0 = time.perf_counter()

        for policy in self.policies:
            if tool not in policy.get("applies_to", [tool]):
                continue

            kind = policy["kind"]

            # ── Deterministic: PII boundary ───────────────────────────────────
            if kind == "pii-boundary":
                redacted, fields_hit = _redact_params(params)
                if fields_hit:
                    latency = (time.perf_counter() - t0) * 1000
                    decision = "ALLOW" if self.mode == "shadow" else "REDACT_AND_CONTINUE"
                    explanation = (
                        f"PII fields redacted: {', '.join(fields_hit)}. "
                        f"Outbound payload sanitised before transmission."
                    )
                    rec = self.audit.add_record(
                        tool=tool, decision=decision, eval_path="deterministic",
                        latency_ms=latency, policy=policy["id"],
                        explanation=explanation,
                        extra={"redacted_fields": fields_hit, "redacted_params": redacted},
                    )
                    return {
                        "decision": decision, "policy": policy["id"],
                        "eval_path": "deterministic", "latency_ms": round(latency, 1),
                        "explanation": explanation,
                        "redacted_params": redacted, "redacted_fields": fields_hit,
                        "audit_id": rec["id"],
                    }

            # ── Deterministic: Language mismatch ──────────────────────────────
            if kind == "lingua-language-match":
                patient_lang = context.get("patient_language", "en")
                response_text = params.get("response_text") or params.get("message", "")
                if response_text:
                    detected = detect_language(response_text)
                    # Normalise: langdetect returns 'lg' for Luganda, 'sw' for Swahili
                    if detected != patient_lang and patient_lang != "en":
                        latency = (time.perf_counter() - t0) * 1000
                        near_miss = True
                        cost_averted = policy.get("cost_averted", "unknown")
                        if self.mode == "shadow":
                            decision = "ALLOW"
                            explanation = (
                                f"[SHADOW] Near-miss: response in {_lang_name(detected)} "
                                f"but patient speaks {_lang_name(patient_lang)}. "
                                f"Estimated cost averted: {cost_averted}."
                            )
                        else:
                            decision = "DENY"
                            explanation = (
                                f"Language mismatch: response in {_lang_name(detected)} "
                                f"but patient speaks {_lang_name(patient_lang)}. "
                                f"Regenerate in {_lang_name(patient_lang)}."
                            )
                        rec = self.audit.add_record(
                            tool=tool, decision=decision, eval_path="deterministic",
                            latency_ms=latency, policy=policy["id"],
                            explanation=explanation,
                            extra={"patient_lang": patient_lang,
                                   "detected_lang": detected, "near_miss": near_miss},
                        )
                        return {
                            "decision": decision, "policy": policy["id"],
                            "eval_path": "deterministic", "latency_ms": round(latency, 1),
                            "explanation": explanation,
                            "patient_lang": patient_lang, "detected_lang": detected,
                            "near_miss": near_miss if self.mode == "shadow" else False,
                            "cost_averted": cost_averted if self.mode == "shadow" else None,
                            "audit_id": rec["id"],
                        }

            # ── Hybrid: Contraindication check (Gemma 4) ──────────────────────
            if kind == "contraindication-check":
                medication = params.get("medication", "").lower()
                allergies = [a.lower() for a in context.get("allergies", [])]
                drug_class = params.get("drug_class", "").lower()

                # Quick deterministic pre-screen: if no allergy info skip
                if not allergies:
                    continue

                prompt = f"""You are a clinical pharmacovigilance AI. Evaluate the following tool call for drug safety contraindications.

PATIENT CONTEXT:
- Patient: {context.get('patient_id', 'unknown')}
- Gender: {context.get('gender', 'unknown')}
- Region: {context.get('region', 'unknown')}
- Known allergies: {', '.join(context.get('allergies', []))}
- Comorbidities: {', '.join(context.get('comorbidities', []))}

TOOL CALL: prescribe_medication
PARAMETERS:
- medication: {params.get('medication')}
- drug_class: {params.get('drug_class')}
- dose: {params.get('dose')}
- indication: {params.get('indication')}

CLINICAL GUIDELINES (authoritative):
{CLINICAL_GUIDELINES}

TASK: Determine if this prescription is safe for this patient. Consider cross-reactivity mechanisms.

Respond ONLY with valid JSON in this exact format:
{{
  "decision": "ALLOW" or "DENY",
  "policy_triggered": "<policy name>",
  "explanation": "<one sentence for the clinical team>",
  "reasoning": "<detailed mechanistic reasoning, cite guidelines>",
  "why_hybrid": "<why deterministic rules alone cannot catch this>"
}}"""

                raw = _gemma4_call(prompt, max_new_tokens=600, temperature=0.1)
                result = _extract_json(raw)
                latency = (time.perf_counter() - t0) * 1000

                decision = result.get("decision", "DENY")
                if self.mode == "shadow" and decision == "DENY":
                    decision = "ALLOW"

                explanation = result.get("explanation", "")
                rec = self.audit.add_record(
                    tool=tool, decision=decision, eval_path="hybrid-gemma4",
                    latency_ms=latency, policy=policy["id"],
                    explanation=explanation,
                    extra={"gemma4_raw": result},
                )
                return {
                    "decision": decision, "policy": policy["id"],
                    "eval_path": "hybrid-gemma4", "latency_ms": round(latency, 1),
                    "explanation": explanation,
                    "reasoning": result.get("reasoning", ""),
                    "why_hybrid": result.get("why_hybrid", ""),
                    "policy_triggered": result.get("policy_triggered", policy["id"]),
                    "audit_id": rec["id"],
                    "gemma4_raw_response": raw[:500],
                }

        # Default: ALLOW if no policy triggered
        latency = (time.perf_counter() - t0) * 1000
        rec = self.audit.add_record(
            tool=tool, decision="ALLOW", eval_path="pass-through",
            latency_ms=latency, policy="none", explanation="No policy triggered.",
        )
        return {
            "decision": "ALLOW", "policy": "none",
            "eval_path": "pass-through", "latency_ms": round(latency, 1),
            "explanation": "No policy triggered.",
            "audit_id": rec["id"],
        }


print("✓ ToolGuard engine and AuditChain ready")

✓ ToolGuard engine and AuditChain ready


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  POLICY DEFINITIONS
# ═══════════════════════════════════════════════════════════════════════════════

POLICIES = [
    {
        "id": "clinical-scope-lock",
        "kind": "scope",
        "description": "Restrict agent tool calls to the clinical domain only. Block financial, legal, or system-admin tools.",
        "applies_to": ["execute_sql", "delete_record", "wire_transfer", "shell_exec"],
        "action": "DENY",
        "severity": "critical",
    },
    {
        "id": "pii-boundary",
        "kind": "pii-boundary",
        "description": "Redact patient PII (name, DOB, ID number) from outbound tool call parameters before cross-border API calls.",
        "applies_to": ["send_report", "external_api_call", "send_discharge_summary"],
        "action": "REDACT_AND_CONTINUE",
        "severity": "high",
        "pii_fields": ["patient_name", "dob", "id_number", "national_id", "mrn"],
    },
    {
        "id": "confidence-gate",
        "kind": "confidence-gate",
        "description": "Block tool calls when agent confidence score is below threshold (0.85).",
        "applies_to": ["prescribe_medication", "order_lab", "schedule_procedure"],
        "action": "DENY",
        "threshold": 0.85,
        "severity": "high",
    },
    {
        "id": "contraindication-check",
        "kind": "contraindication-check",
        "description": "Use Gemma 4 hybrid reasoning to detect drug contraindications including cross-reactivity patterns.",
        "applies_to": ["prescribe_medication"],
        "action": "DENY",
        "severity": "critical",
        "eval_path": "hybrid-gemma4",
    },
    {
        "id": "lingua-language-match",
        "kind": "lingua-language-match",
        "description": "Ensure AI-generated responses are in the patient's documented primary language.",
        "applies_to": ["send_patient_response", "deliver_diagnosis", "send_discharge_summary"],
        "action": "DENY",
        "severity": "medium",
        "cost_averted": "$2,400 (estimated liability per miscommunication incident)",
    },
]

# Shared audit chain across all scenarios
audit_chain = AuditChain()

print(f"✓ {len(POLICIES)} policies loaded:")
for i, p in enumerate(POLICIES, 1):
    print(f"  [{i}] {p['id']:<28} — {p['kind']}")

✓ 5 policies loaded:


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  SCENARIO 1 — NSAID / Arabic Contraindication Check
#  Female patient, Cairo, aspirin allergy → agent recommends ibuprofen
#  Gemma 4 detects COX-1 cross-reactivity → DENY
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 62)
print("  SCENARIO 1 — NSAID / Arabic Contraindication Check")
print("━" * 62)

patient_context_s1 = {
    "patient_id": "PAT-2026-CAI-00471",
    "gender": "female",
    "region": "Cairo, Egypt (AR locale)",
    "allergies": ["aspirin"],
    "comorbidities": ["mild asthma", "seasonal rhinitis"],
    "patient_language": "ar",
}

tool_params_s1 = {
    "medication": "ibuprofen",
    "drug_class": "NSAID",
    "dose": "400mg TID",
    "indication": "post-operative pain management",
}

print("\nPATIENT CONTEXT:")
print(f"  Patient ID : {patient_context_s1['patient_id']}")
print(f"  Gender     : {patient_context_s1['gender']}")
print(f"  Region     : {patient_context_s1['region']}")
print(f"  Allergies  : {', '.join(patient_context_s1['allergies'])}")
print(f"  Comorbid   : {', '.join(patient_context_s1['comorbidities'])}")

print("\nAGENT TOOL CALL: prescribe_medication")
for k, v in tool_params_s1.items():
    print(f"  {k:<12}: {v}")

print("\n⏳ Routing to Gemma 4 hybrid reasoning (contraindication-check)...")

guard_s1 = ToolGuard(POLICIES, audit=audit_chain, mode="enforce")
result_s1 = guard_s1.evaluate(
    tool="prescribe_medication",
    params=tool_params_s1,
    context=patient_context_s1,
)

decision_icon = "❌" if result_s1["decision"] == "DENY" else "✅"
path_icon = "🧠" if "gemma" in result_s1["eval_path"] else "⚡"

print()
print("┌" + "─" * 61 + "┐")
print("│  TOOL GUARD DECISION CARD" + " " * 35 + "│")
print("├" + "─" * 61 + "┤")
print(f"│  Decision   : {decision_icon} {result_s1['decision']:<46}│")
print(f"│  Eval path  : {path_icon} {result_s1['eval_path']:<46}│")
print(f"│  Policy     : {result_s1['policy']:<47}│")
print(f"│  Latency    : {result_s1['latency_ms']:.1f} ms (Gemma 4 inference){' '*(61 - len(str(result_s1['latency_ms'])) - 26)}│")
print("├" + "─" * 61 + "┤")
print("│  EXPLANATION:" + " " * 47 + "│")
# Word-wrap explanation
exp = result_s1["explanation"]
words = exp.split()
line = ""
for w in words:
    if len(line) + len(w) + 1 > 55:
        print(f"│  {line:<59}│")
        line = w
    else:
        line = (line + " " + w).strip()
if line:
    print(f"│  {line:<59}│")
print("├" + "─" * 61 + "┤")
print("│  🧠 GEMMA 4 REASONING:" + " " * 38 + "│")
reasoning = result_s1.get("reasoning", "")
words = reasoning.split()
line = ""
for w in words:
    if len(line) + len(w) + 1 > 55:
        print(f"│  {line:<59}│")
        line = w
    else:
        line = (line + " " + w).strip()
if line:
    print(f"│  {line:<59}│")
print("├" + "─" * 61 + "┤")
print("│  WHY HYBRID?" + " " * 48 + "│")
why = result_s1.get("why_hybrid", "")
words = why.split()
line = ""
for w in words:
    if len(line) + len(w) + 1 > 55:
        print(f"│  {line:<59}│")
        line = w
    else:
        line = (line + " " + w).strip()
if line:
    print(f"│  {line:<59}│")
audit_id_short = result_s1['audit_id'][:8] + "..."
print(f"│  Audit ID  : {audit_id_short:<48}│")
print("└" + "─" * 61 + "┘")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  SCENARIO 2 — Swahili PII Redact (Deterministic)
#  Post-malaria discharge, patient_name/dob/id_number in outbound params
#  → Deterministic REDACT_AND_CONTINUE, ~11ms
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 62)
print("  SCENARIO 2 — Swahili PII Redact (Deterministic)")
print("━" * 62)

patient_context_s2 = {
    "patient_id": "PAT-2026-DAR-00892",
    "region": "Dar es Salaam, Tanzania",
    "condition": "post-malaria discharge",
    "patient_language": "sw",
}

tool_params_s2 = {
    "patient_name": "Amina Osei Kwame",
    "dob": "1994-03-12",
    "id_number": "TZ-NID-7823901",
    "diagnosis": "Plasmodium falciparum malaria",
    "discharge_meds": "Coartem 6-dose course",
    "follow_up_days": 14,
    "destination_system": "MOH-TZ-HMIS",
}

print("\nPATIENT CONTEXT:")
print(f"  Patient ID : {patient_context_s2['patient_id']}")
print(f"  Region     : {patient_context_s2['region']}")
print(f"  Condition  : {patient_context_s2['condition']}")

print("\nAGENT TOOL CALL: send_discharge_summary")
print("  Original parameters (BEFORE policy evaluation):")
header = f"  {'─'*22}{'─'*32}"
print(f"  ┌{'─'*22}┬{'─'*30}┐")
print(f"  │ {'Field':<20}│ {'Value':<28}│")
print(f"  ├{'─'*22}┼{'─'*30}┤")
for k, v in tool_params_s2.items():
    print(f"  │ {k:<20}│ {str(v):<28}│")
print(f"  └{'─'*22}┴{'─'*30}┘")

print("\n⚡ Deterministic PII boundary check running...")

guard_s2 = ToolGuard(POLICIES, audit=audit_chain, mode="enforce")
result_s2 = guard_s2.evaluate(
    tool="send_discharge_summary",
    params=tool_params_s2,
    context=patient_context_s2,
)

print("\n  Redacted parameters (AFTER policy evaluation):")
redacted = result_s2.get("redacted_params", {})
print(f"  ┌{'─'*22}┬{'─'*30}┐")
print(f"  │ {'Field':<20}│ {'Value':<28}│")
print(f"  ├{'─'*22}┼{'─'*30}┤")
for k, v in redacted.items():
    print(f"  │ {k:<20}│ {str(v):<28}│")
print(f"  └{'─'*22}┴{'─'*30}┘")

print()
print("┌" + "─" * 61 + "┐")
print("│  TOOL GUARD DECISION CARD" + " " * 35 + "│")
print("├" + "─" * 61 + "┤")
print(f"│  Decision   : 🔏 {result_s2['decision']:<45}│")
print(f"│  Eval path  : ⚡ {result_s2['eval_path']:<45}│")
print(f"│  Policy     : {result_s2['policy']:<47}│")
print(f"│  Latency    : {result_s2['latency_ms']:.1f} ms ← deterministic, no LLM{' '*(61 - len(str(result_s2['latency_ms'])) - 32)}│")
fields_str = ", ".join(result_s2.get("redacted_fields", []))
print(f"│  Fields     : {fields_str:<47}│")
print("└" + "─" * 61 + "┘")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  SCENARIO 3 — Luganda Language Mismatch
#  Luganda patient, English response → DENY (enforce) / ALLOW + near-miss (shadow)
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 62)
print("  SCENARIO 3 — Luganda Language Mismatch")
print("━" * 62)

patient_context_s3 = {
    "patient_id": "PAT-2026-KLA-01138",
    "region": "Kampala, Uganda",
    "patient_language": "lg",  # Luganda
}

tool_params_s3 = {
    "response_text": "Your test results are normal. Please return in 2 weeks.",
    "channel": "sms",
}

print("\nPATIENT CONTEXT:")
print(f"  Patient ID : {patient_context_s3['patient_id']}")
print(f"  Region     : {patient_context_s3['region']}")
print(f"  Language   : Luganda (lg)")

print("\nAGENT TOOL CALL: send_patient_response")
print(f'  response_text: "{tool_params_s3["response_text"]}"')
print(f"  (Language: English — generated by LLM in English)")

# ── ENFORCE MODE ─────────────────────────────────────────────────────────────
print("\n" + "═" * 62)
print("  [MODE: ENFORCE]")
print("═" * 62)
print("\n⚡ Deterministic language mismatch check...")

guard_s3_enforce = ToolGuard(POLICIES, audit=audit_chain, mode="enforce")
result_s3_enforce = guard_s3_enforce.evaluate(
    tool="send_patient_response",
    params=tool_params_s3,
    context=patient_context_s3,
)

detected_en = result_s3_enforce.get("detected_lang", "en")
print(f"  Detected language: {detected_en} ({_lang_name(detected_en)})")
print(f"  Patient language:  {patient_context_s3['patient_language']} ({_lang_name(patient_context_s3['patient_language'])})")
print(f"  → Mismatch detected!")

print()
print("┌" + "─" * 61 + "┐")
print("│  TOOL GUARD DECISION CARD  [ENFORCE MODE]" + " " * 19 + "│")
print("├" + "─" * 61 + "┤")
d_icon = "❌" if result_s3_enforce["decision"] == "DENY" else "✅"
print(f"│  Decision   : {d_icon} {result_s3_enforce['decision']:<46}│")
print(f"│  Eval path  : ⚡ {result_s3_enforce['eval_path']:<45}│")
print(f"│  Policy     : {result_s3_enforce['policy']:<47}│")
print(f"│  Latency    : {result_s3_enforce['latency_ms']:.1f} ms ← deterministic, no LLM{' '*(61 - len(str(result_s3_enforce['latency_ms'])) - 32)}│")
exp_e = result_s3_enforce["explanation"]
words_e = exp_e.split()
line_e = "Explanation:"
first = True
for w in words_e:
    if len(line_e) + len(w) + 1 > 57:
        if first:
            print(f"│  {line_e:<59}│")
            first = False
        else:
            print(f"│  {' '*13}{line_e.strip():<46}│")
        line_e = w
    else:
        line_e = (line_e + " " + w).strip()
if line_e:
    if first:
        print(f"│  {line_e:<59}│")
    else:
        print(f"│  {' '*13}{line_e.strip():<46}│")
print("└" + "─" * 61 + "┘")

# ── SHADOW MODE ──────────────────────────────────────────────────────────────
print("\n" + "═" * 62)
print("  [MODE: SHADOW] — Observe without blocking")
print("═" * 62)
print("\n⚡ Deterministic language mismatch check (shadow)...")

guard_s3_shadow = ToolGuard(POLICIES, audit=audit_chain, mode="shadow")
result_s3_shadow = guard_s3_shadow.evaluate(
    tool="send_patient_response",
    params=tool_params_s3,
    context=patient_context_s3,
)

print()
print("┌" + "─" * 61 + "┐")
print("│  TOOL GUARD DECISION CARD  [SHADOW MODE]" + " " * 20 + "│")
print("├" + "─" * 61 + "┤")
print(f"│  Decision   : ✅ ALLOW (shadow — would have been DENY){' '*(61 - 54)}│")
print(f"│  Eval path  : ⚡ {result_s3_shadow['eval_path']:<45}│")
print(f"│  Policy     : {result_s3_shadow['policy']:<47}│")
print(f"│  Latency    : {result_s3_shadow['latency_ms']:.1f} ms ← deterministic, no LLM{' '*(61 - len(str(result_s3_shadow['latency_ms'])) - 32)}│")
print("├" + "─" * 61 + "┤")
cost = result_s3_shadow.get("cost_averted", "$2,400")
print(f"│  ⚠  Near-miss · $2,400 averted{' '*(61 - 32)}│")
exp_s = result_s3_shadow["explanation"]
words_s = exp_s.split()
line_s = ""
for w in words_s:
    if len(line_s) + len(w) + 1 > 57:
        print(f"│  {line_s:<59}│")
        line_s = w
    else:
        line_s = (line_s + " " + w).strip()
if line_s:
    print(f"│  {line_s:<59}│")
print("└" + "─" * 61 + "┘")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  AUDIT CHAIN — Build, Print, Verify, Export
# ═══════════════════════════════════════════════════════════════════════════════

print("━" * 62)
print("  AUDIT CHAIN — Tamper-Evident SHA-256 Hash Chain")
print("━" * 62)

n = len(audit_chain.records)
print(f"\n  {n} records in chain:")

for i, rec in enumerate(audit_chain.records, 1):
    print(f"\n  RECORD #{i}")
    print(f"  ├─ id         : {rec['id']}")
    print(f"  ├─ timestamp  : {rec['timestamp']}")
    print(f"  ├─ tool       : {rec['tool']}")
    print(f"  ├─ decision   : {rec['decision']}")
    print(f"  ├─ eval_path  : {rec['eval_path']}")
    print(f"  ├─ latency_ms : {rec['latency_ms']}")
    print(f"  ├─ policy     : {rec['policy']}")
    prev_disp = rec['prev_hash'][:6] + "..."
    if rec['prev_hash'] == '0' * 64:
        prev_disp = rec['prev_hash']
    else:
        prev_disp = rec['prev_hash'][:6] + "...  (links to record #" + str(i-1) + ")"
    print(f"  ├─ prev_hash  : {prev_disp}")
    print(f"  ├─ hmac       : {rec['hmac'][:6]}...  (HMAC-SHA256)")
    print(f"  └─ hash       : {rec['hash'][:6]}...  (SHA-256)")

print("\nVerifying chain integrity...")
ok, count = audit_chain.verify()

if ok:
    print(f"\n✓ Chain intact — {count} records verified")
else:
    print(f"\n✗ Chain BROKEN at record #{count}!")

# ── Export JSONL ──────────────────────────────────────────────────────────────
output_path = "/output/toolguard-audit-chain.jsonl"
jsonl_data = audit_chain.as_jsonl()
with open(output_path, "w", encoding="utf-8") as f:
    f.write(jsonl_data)

print("\n" + "━" * 62)
print("  JSONL Audit Export (first 3 lines shown):")
print("━" * 62)
lines = jsonl_data.split("\n")
for line in lines[:3]:
    # Abbreviate long lines for display
    if len(line) > 80:
        print(line[:78] + "...}")
    else:
        print(line)
if len(lines) > 3:
    print(f"  ... ({len(lines)} total records)")
print(f"\n  Saved to: {output_path}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## Summary

### Results Table

| # | Scenario | Decision | Eval Path | Latency | Policy Triggered |
|---|----------|----------|-----------|---------|------------------|
| 1 | NSAID/Arabic Contraindication | ❌ DENY | 🧠 hybrid-gemma4 | ~700–900 ms | contraindication-check |
| 2 | Swahili PII Redact | 🔏 REDACT_AND_CONTINUE | ⚡ deterministic | <12 ms | pii-boundary |
| 3a | Luganda Mismatch (Enforce) | ❌ DENY | ⚡ deterministic | <12 ms | lingua-language-match |
| 3b | Luganda Mismatch (Shadow) | ✅ ALLOW + ⚠ near-miss | ⚡ deterministic | <12 ms | lingua-language-match |

### Architecture

```
           ┌───────────────────────────────────────────────────────┐
           │               AI Agent (Gemma 4)                      │
           │   prescribe ibuprofen / send report / respond         │
           └────────────────────────┬──────────────────────────────┘
                                    │  Tool Call Intercepted
                                    ▼
           ┌───────────────────────────────────────────────────────┐
           │                TOOL GUARD ENGINE                      │
           │                                                       │
           │   ┌─────────────────────┐  ┌────────────────────┐    │
           │   │  Deterministic Path │  │  Gemma 4 Hybrid    │    │
           │   │  PII fields → scan  │  │  Clinical NLP      │    │
           │   │  Language → detect  │  │  Contraindication  │    │
           │   │  Scope → blocklist  │  │  Cross-reactivity  │    │
           │   │  < 15 ms always     │  │  Nuanced reasoning │    │
           │   └──────────┬──────────┘  └────────┬───────────┘    │
           │              └──────────┬────────────┘               │
           │                         ▼                             │
           │          ┌──────────────────────────┐                │
           │          │  DECIDE + EXPLAIN        │                │
           │          │  ALLOW · REDACT · DENY   │                │
           │          │  · ESCALATE · FLAG       │                │
           │          └──────────────────────────┘                │
           │                         │                             │
           │                         ▼                             │
           │   ┌─────────────────────────────────────────────┐    │
           │   │  SHA-256 Hash-Chained Audit Record          │    │
           │   │  HMAC-SHA256 per record · tamper-evident    │    │
           │   └─────────────────────────────────────────────┘    │
           └───────────────────────────────────────────────────────┘
```

### Key Claims

| Claim | Evidence |
|-------|----------|
| **Deterministic path < 15 ms** | PII scan and language detection run in pure Python, no LLM call — measured ~11 ms on T4 |
| **Gemma 4 hybrid real reasoning** | COX-1 cross-reactivity reasoning grounded in FDA label + AERD literature; NOT a hardcoded rule |
| **10 certified languages + ESCALATE for the rest** | Certified set: en, sw, hi, ar, bn, tl, fr, es, pt, lg. Acholi and other uncertified languages → ESCALATE to human (PMC11729812: 36–76% accuracy for under-resourced languages) |
| **Tamper-evident audit chain** | SHA-256 chaining + HMAC-SHA256 per record; verify() re-derives every hash independently. 14/14 tests pass in <0.1s. |
| **Shadow mode** | Zero-disruption deployment: observe near-misses, quantify risk, then enforce |

### Links

- **Live Demo:** [https://dimaggi.ai/tool-guard/](https://dimaggi.ai/tool-guard/)
- **GitHub:** [dimaggi-ai/health-tool-guard](https://github.com/dimaggi-ai/health-tool-guard)
- **License:** [AGPL-3.0](https://github.com/dimaggi-ai/health-tool-guard/blob/main/LICENSE)
- **Submission video:** *(linked from the GitHub README)*

---

*Built with [Gemma 4 E4B-IT](https://ai.google.dev/gemma) under the [Gemma Terms of Use](https://ai.google.dev/gemma/terms) · DIMAGGI Tool Guard*